# Wire three tasks into a Lakeflow Job with a conditional branch, retries, and a deliberately failing run that surfaces in run history

The team's PM has stopped trusting the morning dashboard because every Tuesday it shows zeros for four hours before the on-call notices ingest failed at midnight. The fix is two things at once: retries on the unreliable task, and a notify branch when the validation row count slips below 1000. You have one session to wire both into a Lakeflow Job, fire a deliberately failing run, and prove the run history view makes the failure obvious to the next on-call who looks at it without your help.

The hands-on work:

- Create a UC schema `workspace.default.labrun_jobs`, a UC Volume for source files, three staging tables.
- Author four task notebooks (`ingest.py`, `transform.py`, `validate.py`, `notify.py`) in a workspace folder.
- Create a Lakeflow Job with four tasks wired as a DAG: `transform` depends on `ingest`, `validate` depends on `transform`, `notify_on_low_count` depends on `validate` via an `if_else_condition` task that fires only when validate's row count is below 1000.
- Configure the ingest task with `max_retries=2` and `min_retry_interval_millis=30000`.
- Trigger a run with a parameter that forces the first ingest attempt to fail. The retry succeeds; transform and validate run; the conditional branch fires (the seed file is deliberately small so row_count < 1000).
- Read the run history view and confirm the retry, the branch decision, and the SUCCESS state are all visible.

**Time.** Plan for about 60 minutes hands-on. The deliberate-failure run takes 3 to 6 minutes of wall-clock including the 30-second retry backoff. Budget up to 110 minutes for the session window.

**Cost.** Zero dollars. One full job run including the deliberate retry is the chunkiest single-run spend in the Databricks DE Associate cert track besides Lab 6.

This lab maps to Databricks DE Associate Domain 4 (Lakeflow Jobs, ~10%) and Domain 6 (Troubleshooting and Monitoring, ~8% provisional).

**Rename callout.** If your other prep material says "Workflows" or "Databricks Jobs" it means Lakeflow Jobs on the current exam. The API namespace is still `/api/2.1/jobs`. If it says "Databricks Repos" it means Databricks Git Folders.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Job IDs are captured at create time; the
# four task notebooks live in a workspace folder owned by the caller.

import atexit
import base64
import getpass
import io
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from databricks.sdk.service import jobs as j_svc
from databricks.sdk.service.workspace import ImportFormat, Language, ExportFormat

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "lakeflow-jobs-orchestration-with-control-flow"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_jobs"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

SOURCE_VOLUME = "source_files"
SOURCE_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{SOURCE_VOLUME}"
SOURCE_VOLUME_PATH = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{SOURCE_VOLUME}"

STAGING_TABLE = "staging"
VALIDATION_TABLE = "validation_log"
NOTIFY_TABLE = "notify_log"
STAGING_FQN = f"{LAB_SCHEMA_FQN}.{STAGING_TABLE}"
VALIDATION_FQN = f"{LAB_SCHEMA_FQN}.{VALIDATION_TABLE}"
NOTIFY_FQN = f"{LAB_SCHEMA_FQN}.{NOTIFY_TABLE}"

JOB_NAME = "labrun-jobs-control-flow"
JOB_ID = None  # set in Task 2 after job create
JOB_FOLDER = None  # set in Task 2 once caller is known
INGEST_NB = None
TRANSFORM_NB = None
VALIDATE_NB = None
NOTIFY_NB = None

# Notebook bodies. Each is a Databricks notebook source file. The first line
# `# Databricks notebook source` is the marker Databricks recognizes.
INGEST_SOURCE = f"""# Databricks notebook source
# Ingest task. Reads seed file from a UC Volume and writes to staging.
# Honors the force_fail parameter: when force_fail==1 the task raises on the
# first attempt. The second attempt uses dbutils.jobs.taskValues to read the
# attempt number from the job runtime, which is 0-indexed; on attempt 0 we
# fail, on attempt 1 we succeed.

dbutils.widgets.text("force_fail", "0")
dbutils.widgets.text("attempt_signal", "")

force_fail = dbutils.widgets.get("force_fail") == "1"

# The job runtime sets a context property the notebook can read to know its
# current attempt. The pattern is: on attempt > 0 we always succeed; on
# attempt 0 we honor force_fail.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
try:
    attempt = int(ctx.attributes().get("attemptNumber").get())
except Exception:
    attempt = 0

print(f"ingest attempt: {{attempt}} force_fail={{force_fail}}")

if force_fail and attempt == 0:
    raise RuntimeError("Deliberate ingest failure on attempt 0 (force_fail=1)")

source_path = f"{SOURCE_VOLUME_PATH}/seed.json"
df = spark.read.format("json").load(source_path)
df.write.mode("overwrite").saveAsTable("{STAGING_FQN}")
print(f"ingest wrote {{df.count()}} rows to {STAGING_FQN}")
"""

TRANSFORM_SOURCE = f"""# Databricks notebook source
# Transform task. Reads staging, lowercases the source_id field, writes
# back to the same table. Keeps the task minimal; the lab's lesson is the
# DAG, not the transformation.

from pyspark.sql import functions as F

df = spark.read.table("{STAGING_FQN}")
df = df.withColumn("source_id", F.lower(F.col("source_id")))
df.write.mode("overwrite").saveAsTable("{STAGING_FQN}")
print(f"transform wrote {{df.count()}} rows")
"""

VALIDATE_SOURCE = f"""# Databricks notebook source
# Validate task. Counts rows in staging, writes the count to validation_log,
# emits the row_count as a task value so the conditional branch can read it.

from pyspark.sql import functions as F
from datetime import datetime, timezone

row_count = spark.read.table("{STAGING_FQN}").count()
row = [(datetime.now(timezone.utc).isoformat(), int(row_count))]
schema = "validated_at STRING, row_count INT"
df = spark.createDataFrame(row, schema)
df.write.mode("append").saveAsTable("{VALIDATION_FQN}")

dbutils.jobs.taskValues.set(key="row_count", value=int(row_count))
print(f"validate row_count={{row_count}}")
"""

NOTIFY_SOURCE = f"""# Databricks notebook source
# Notify task. Writes one row into notify_log so we can prove the
# conditional branch fired. No email send on Free Edition.

from datetime import datetime, timezone

row = [(datetime.now(timezone.utc).isoformat(), "row_count < 1000; please investigate")]
schema = "noticed_at STRING, message STRING"
df = spark.createDataFrame(row, schema)
df.write.mode("append").saveAsTable("{NOTIFY_FQN}")
print("notify fired")
"""

# Tiny seed file (well below 1000 rows so the conditional branch fires).
SEED_ROWS = [
    {{"source_id": f"S-{i:03d}", "amount": i * 5.0}}
    for i in range(1, 11)
]

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials, resolve the
# Starter SQL warehouse, set up the workspace job folder path.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


JOB_FOLDER = f"/Workspace/Users/{CALLER_USER}/labrun-jobs-control-flow"
INGEST_NB = f"{JOB_FOLDER}/ingest"
TRANSFORM_NB = f"{JOB_FOLDER}/transform"
VALIDATE_NB = f"{JOB_FOLDER}/validate"
NOTIFY_NB = f"{JOB_FOLDER}/notify"
print(f"Job folder will be: {JOB_FOLDER}")

register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. The Lakeflow Job is
# critical=True; cleanup cancels in-flight runs, then deletes the job.


def _find_jobs_by_name(name):
    found = []
    try:
        for job in w.jobs.list(name=name):
            if job.settings and job.settings.name == name:
                found.append(job)
    except Exception:
        pass
    return found


CLEANUP_MANIFEST = [
    CleanupResource(
        type="lakeflow_job",
        id=JOB_NAME,
        region="databricks",
        critical=True,
        cli_delete_command=f"databricks jobs delete --name {JOB_NAME}",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=NOTIFY_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {NOTIFY_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=VALIDATION_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {VALIDATION_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=STAGING_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {STAGING_FQN}\"",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:{SOURCE_VOLUME_PATH}/",
    ),
    CleanupResource(
        type="uc_volume",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {SOURCE_VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="workspace_path",
        id=JOB_FOLDER,
        region="databricks",
        cli_delete_command=f"databricks workspace delete --recursive {JOB_FOLDER}",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "workspace_path":
            try:
                w.workspace.delete(rid, recursive=True)
            except Exception:
                pass
        elif rtype == "lakeflow_job":
            for job in _find_jobs_by_name(rid):
                try:
                    # Cancel any in-flight runs first.
                    runs = w.jobs.list_runs(job_id=job.job_id, active_only=True)
                    for run in (runs.runs or []):
                        try:
                            w.jobs.cancel_run(run.run_id)
                        except Exception:
                            pass
                except Exception:
                    pass
                try:
                    w.jobs.delete(job.job_id)
                except Exception:
                    pass
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                return len(list(w.files.list_directory_contents(volume_path))) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "workspace_path":
            try:
                w.workspace.get_status(rid)
                return True
            except Exception:
                return False
        if rtype == "lakeflow_job":
            return len(_find_jobs_by_name(rid)) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        # Jobs API tag scan.
        try:
            for job in w.jobs.list():
                tags = (job.settings.tags if job.settings else None) or {}
                if tags.get(LAB_TAG_KEY) == lab_slug:
                    arns.append(f"job:{job.settings.name}")
        except Exception:
            pass
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Schema, volume, seed file, four task notebooks

Six things to do, all idempotent:

1. `CREATE SCHEMA workspace.default.labrun_jobs` and tag it.
2. `CREATE VOLUME labrun_jobs.source_files` and tag it.
3. Upload a 10-row `seed.json` file (ndjson) into the volume. The 10 rows guarantee the conditional branch fires (row_count is well below 1000).
4. Create the workspace folder for the four notebooks.
5. Upload the four notebooks (`ingest`, `transform`, `validate`, `notify`) using the `INGEST_SOURCE`, `TRANSFORM_SOURCE`, `VALIDATE_SOURCE`, `NOTIFY_SOURCE` constants from the imports cell.
6. Pre-create the three managed tables (`staging`, `validation_log`, `notify_log`) so the validator can read row counts before the first job run.

In [ ]:
# NBVAL_SKIP
# Task 1: Schema, volume, seed file, workspace folder, four notebooks,
# three pre-created tables.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Tag the schema with ALTER SCHEMA LAB_SCHEMA_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE).

# YOUR CODE: Run CREATE VOLUME IF NOT EXISTS SOURCE_VOLUME_FQN.

# YOUR CODE: Tag the volume.

ndjson_payload = ("\n".join(json.dumps(row) for row in SEED_ROWS) + "\n").encode("utf-8")
# YOUR CODE: Upload ndjson_payload to f"{SOURCE_VOLUME_PATH}/seed.json"
# via w.files.upload with overwrite=True.

# Create the workspace folder for the four task notebooks.
try:
    w.workspace.mkdirs(JOB_FOLDER)
except Exception:
    pass

NOTEBOOKS = [
    (INGEST_NB, INGEST_SOURCE, "ingest"),
    (TRANSFORM_NB, TRANSFORM_SOURCE, "transform"),
    (VALIDATE_NB, VALIDATE_SOURCE, "validate"),
    (NOTIFY_NB, NOTIFY_SOURCE, "notify"),
]
for path, src, label in NOTEBOOKS:
    # YOUR CODE: Call w.workspace.import_(path=path, format=ImportFormat.SOURCE,
    # language=Language.PYTHON, content=base64.b64encode(src.encode()).decode(),
    # overwrite=True). The .py is treated as a Databricks notebook because the
    # first line is `# Databricks notebook source`.
    print(f"Wrote notebook: {path}")

# Pre-create the three managed tables so Checkpoint 3's notify_log lookup
# does not blow up before the first job run if the conditional did not fire.
for fqn, schema in [
    (STAGING_FQN, "source_id STRING, amount DOUBLE"),
    (VALIDATION_FQN, "validated_at STRING, row_count INT"),
    (NOTIFY_FQN, "noticed_at STRING, message STRING"),
]:
    # YOUR CODE: Run CREATE TABLE IF NOT EXISTS fqn (schema) USING DELTA
    # via run_sql().
    # YOUR CODE: Tag each with ALTER TABLE fqn SET TAGS.
    pass

print()
print("Schema, volume, seed file, four task notebooks, three tables: ready.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Schema tagged. Volume tagged. seed.json staged with 10
# rows. Four notebooks exist in the workspace folder. Three tables created.


def checkpoint_1(session):
    try:
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema {LAB_SCHEMA_FQN} not found.",
            )
        tag_sql = (
            "SELECT tag_value FROM system.information_schema.schema_tags "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
            f"AND tag_name = '{LAB_TAG_KEY}'"
        )
        if not any(r[0] == LAB_TAG_VALUE for r in run_sql(tag_sql)["rows"]):
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}.",
            )

        vol_sql = (
            "SELECT 1 FROM system.information_schema.volumes "
            f"WHERE volume_catalog = '{CATALOG}' AND volume_schema = '{LAB_SCHEMA}' "
            f"AND volume_name = '{SOURCE_VOLUME}'"
        )
        if not run_sql(vol_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Volume {SOURCE_VOLUME_FQN} not found.",
            )

        try:
            listing = list(w.files.list_directory_contents(SOURCE_VOLUME_PATH + "/"))
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not list volume contents: {exc}",
            )
        if not any(e.path.endswith("seed.json") for e in listing):
            return CheckpointResult(
                status="fail",
                error_reason="seed.json not found in the source volume.",
            )

        for path, _, label in [
            (INGEST_NB, None, "ingest"),
            (TRANSFORM_NB, None, "transform"),
            (VALIDATE_NB, None, "validate"),
            (NOTIFY_NB, None, "notify"),
        ]:
            try:
                w.workspace.get_status(path)
            except Exception:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Notebook {path} not found. Did the workspace import_ "
                        f"call run for {label}?"
                    ),
                )

        for fqn in (STAGING_FQN, VALIDATION_FQN, NOTIFY_FQN):
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{fqn.split('.')[0]}' "
                f"AND table_schema = '{fqn.split('.')[1]}' "
                f"AND table_name = '{fqn.split('.')[2]}'"
            )
            if not run_sql(sql)["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Table {fqn} not found. Did CREATE TABLE run?",
                )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names the missing piece: schema, schema tag, volume, seed.json, one of the four notebooks, or one of the three tables. Each is a different SQL or SDK call.

</details>

<details><summary>Hint 2 (direction)</summary>

Three groups of work: SQL CREATEs for schema/volume/three tables (plus ALTER for tags), one file upload, four workspace notebook imports. The notebook imports use `w.workspace.import_(path=path, format=ImportFormat.SOURCE, language=Language.PYTHON, content=base64.b64encode(src.encode("utf-8")).decode("utf-8"), overwrite=True)`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
run_sql(f"CREATE VOLUME IF NOT EXISTS {SOURCE_VOLUME_FQN}")
run_sql(f"ALTER VOLUME {SOURCE_VOLUME_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

w.files.upload(
    file_path=f"{SOURCE_VOLUME_PATH}/seed.json",
    contents=io.BytesIO(ndjson_payload),
    overwrite=True,
)

for path, src, _ in NOTEBOOKS:
    w.workspace.import_(
        path=path,
        format=ImportFormat.SOURCE,
        language=Language.PYTHON,
        content=base64.b64encode(src.encode("utf-8")).decode("utf-8"),
        overwrite=True,
    )

for fqn, schema in [
    (STAGING_FQN, "source_id STRING, amount DOUBLE"),
    (VALIDATION_FQN, "validated_at STRING, row_count INT"),
    (NOTIFY_FQN, "noticed_at STRING, message STRING"),
]:
    run_sql(f"CREATE TABLE IF NOT EXISTS {fqn} ({schema}) USING DELTA")
    run_sql(f"ALTER TABLE {fqn} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
```

</details>

**Wallet check.** Still at $0.00. One JSON file, four workspace files, three empty tables. No serverless DBUs yet.

## Task 2: Create the Lakeflow Job with the four-task DAG, retries, and conditional branch

This is the architectural piece. The job has four tasks:

| Task | Type | Depends on | Notes |
| --- | --- | --- | --- |
| `ingest` | notebook | (none) | `max_retries=2`, `min_retry_interval_millis=30000`, parameter `force_fail` |
| `transform` | notebook | ingest | reads/writes staging |
| `validate` | notebook | transform | counts rows, emits row_count task value |
| `notify_on_low_count` | condition (if/else) | validate | fires the notify notebook when `{{tasks.validate.values.row_count}} < 1000` |

The condition task in Jobs API 2.1 is a `condition_task` with `op="LESS_THAN"`, `left="{{tasks.validate.values.row_count}}"`, `right="1000"`. The notify notebook hangs off the true branch (the run_if relationship on a child task; in the SDK this is expressed as a sibling task that depends on `notify_on_low_count` with `run_if=at_least_one_success`, or by setting the notify task itself as the conditional. The lab follows the simpler shape: define a `notify_on_low_count` condition task plus a `notify` notebook task that depends on it with `condition_task` evaluating to TRUE).

The job's `tags` map carries `labrun_lab_slug`. The tag is required for the orphan scan.

In [ ]:
# NBVAL_SKIP
# Task 2: Create the Lakeflow Job.

global JOB_ID

# Build the four tasks. The condition task evaluates row_count < 1000; the
# notify task depends on the condition with run_if=ALL_SUCCESS, meaning it
# runs when the condition is TRUE.

# YOUR CODE: Construct the four task definitions as a list of j_svc.Task
# objects:
#
#   ingest_task = j_svc.Task(
#       task_key="ingest",
#       notebook_task=j_svc.NotebookTask(
#           notebook_path=INGEST_NB,
#           base_parameters={"force_fail": "1"},
#       ),
#       max_retries=2,
#       min_retry_interval_millis=30000,
#       retry_on_timeout=True,
#   )
#
#   transform_task = j_svc.Task(
#       task_key="transform",
#       notebook_task=j_svc.NotebookTask(notebook_path=TRANSFORM_NB),
#       depends_on=[j_svc.TaskDependency(task_key="ingest")],
#   )
#
#   validate_task = j_svc.Task(
#       task_key="validate",
#       notebook_task=j_svc.NotebookTask(notebook_path=VALIDATE_NB),
#       depends_on=[j_svc.TaskDependency(task_key="transform")],
#   )
#
#   notify_branch_task = j_svc.Task(
#       task_key="notify_on_low_count",
#       condition_task=j_svc.ConditionTask(
#           op=j_svc.ConditionTaskOp.LESS_THAN,
#           left="{{tasks.validate.values.row_count}}",
#           right="1000",
#       ),
#       depends_on=[j_svc.TaskDependency(task_key="validate")],
#   )
#
#   notify_task = j_svc.Task(
#       task_key="notify",
#       notebook_task=j_svc.NotebookTask(notebook_path=NOTIFY_NB),
#       depends_on=[j_svc.TaskDependency(task_key="notify_on_low_count", outcome="true")],
#   )
#
# Then call:
#   JOB_ID = w.jobs.create(
#       name=JOB_NAME,
#       tasks=[ingest_task, transform_task, validate_task, notify_branch_task, notify_task],
#       tags={LAB_TAG_KEY: LAB_TAG_VALUE},
#   ).job_id

print(f"Job created: {JOB_NAME} (id={JOB_ID})")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Job exists with the correct DAG, retry config, condition
# expression, and tag.


def checkpoint_2(session):
    try:
        if JOB_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="JOB_ID is None. Did w.jobs.create() run?",
            )
        job = w.jobs.get(JOB_ID)
        if not job or not job.settings:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not fetch job {JOB_ID}.",
            )

        if (job.settings.tags or {}).get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. The orphan "
                    f"scan filters on this tag."
                ),
            )

        tasks_by_key = {t.task_key: t for t in (job.settings.tasks or [])}
        required_keys = {
            "ingest", "transform", "validate", "notify_on_low_count", "notify",
        }
        missing = required_keys - set(tasks_by_key)
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job is missing task keys: {sorted(missing)}. Required: "
                    f"{sorted(required_keys)}."
                ),
            )

        ingest = tasks_by_key["ingest"]
        if (ingest.max_retries or 0) != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ingest.max_retries={ingest.max_retries}; expected 2."
                ),
            )
        if (ingest.min_retry_interval_millis or 0) != 30000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ingest.min_retry_interval_millis="
                    f"{ingest.min_retry_interval_millis}; expected 30000."
                ),
            )

        notify_cond = tasks_by_key["notify_on_low_count"]
        cond = notify_cond.condition_task
        if cond is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "notify_on_low_count is not a condition_task. Use "
                    "j_svc.ConditionTask with op=LESS_THAN, left="
                    "'{{tasks.validate.values.row_count}}', right='1000'."
                ),
            )
        op = (cond.op.value if cond.op else "") or ""
        if "LESS" not in op.upper():
            return CheckpointResult(
                status="fail",
                error_reason=f"Condition op={op!r}; expected LESS_THAN.",
            )
        if "row_count" not in (cond.left or "") or (cond.right or "") != "1000":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Condition operands left={cond.left!r} right={cond.right!r}; "
                    f"expected left referencing 'row_count' and right='1000'."
                ),
            )

        # Dependencies.
        def has_dep(task_key, expected_dep):
            t = tasks_by_key.get(task_key)
            return t and any(
                d.task_key == expected_dep for d in (t.depends_on or [])
            )

        if not has_dep("transform", "ingest"):
            return CheckpointResult(
                status="fail",
                error_reason="transform does not depend on ingest.",
            )
        if not has_dep("validate", "transform"):
            return CheckpointResult(
                status="fail",
                error_reason="validate does not depend on transform.",
            )
        if not has_dep("notify_on_low_count", "validate"):
            return CheckpointResult(
                status="fail",
                error_reason="notify_on_low_count does not depend on validate.",
            )
        if not has_dep("notify", "notify_on_low_count"):
            return CheckpointResult(
                status="fail",
                error_reason="notify does not depend on notify_on_low_count.",
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names which task is missing, which dependency is wrong, or which config value is wrong. Read it before peeking at hint 3.

</details>

<details><summary>Hint 2 (direction)</summary>

Five `j_svc.Task` objects. The ingest task has `max_retries=2`, `min_retry_interval_millis=30000`. The condition task uses `j_svc.ConditionTask(op=j_svc.ConditionTaskOp.LESS_THAN, left="{{tasks.validate.values.row_count}}", right="1000")`. The notify notebook task depends on `notify_on_low_count` with `outcome="true"` on its TaskDependency.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
global JOB_ID

ingest_task = j_svc.Task(
    task_key="ingest",
    notebook_task=j_svc.NotebookTask(
        notebook_path=INGEST_NB,
        base_parameters={"force_fail": "1"},
    ),
    max_retries=2,
    min_retry_interval_millis=30000,
    retry_on_timeout=True,
)

transform_task = j_svc.Task(
    task_key="transform",
    notebook_task=j_svc.NotebookTask(notebook_path=TRANSFORM_NB),
    depends_on=[j_svc.TaskDependency(task_key="ingest")],
)

validate_task = j_svc.Task(
    task_key="validate",
    notebook_task=j_svc.NotebookTask(notebook_path=VALIDATE_NB),
    depends_on=[j_svc.TaskDependency(task_key="transform")],
)

notify_branch_task = j_svc.Task(
    task_key="notify_on_low_count",
    condition_task=j_svc.ConditionTask(
        op=j_svc.ConditionTaskOp.LESS_THAN,
        left="{{tasks.validate.values.row_count}}",
        right="1000",
    ),
    depends_on=[j_svc.TaskDependency(task_key="validate")],
)

notify_task = j_svc.Task(
    task_key="notify",
    notebook_task=j_svc.NotebookTask(notebook_path=NOTIFY_NB),
    depends_on=[
        j_svc.TaskDependency(task_key="notify_on_low_count", outcome="true"),
    ],
)

JOB_ID = w.jobs.create(
    name=JOB_NAME,
    tasks=[ingest_task, transform_task, validate_task, notify_branch_task, notify_task],
    tags={LAB_TAG_KEY: LAB_TAG_VALUE},
).job_id
```

</details>

**Wallet check.** Still at $0.00. Job creation is a metadata operation. The first run is in Task 3; that is where the daily quota takes a real bite.

## Task 3: Trigger the run with the deliberate ingest failure

Trigger the job once. The `force_fail=1` parameter set in the job config means ingest fails on attempt 0 and succeeds on attempt 1. After the 30-second retry backoff, transform and validate run; validate emits `row_count=10` as a task value; the condition `10 < 1000` evaluates TRUE; notify runs and writes one row to `notify_log`.

Expected outcome: `result_state=SUCCESS`, ingest task `attempt_number=2` (one failed plus one successful), notify task ran, `notify_log` has one row.

Use `w.jobs.run_now(job_id=JOB_ID)` then poll `w.jobs.get_run(run_id)` until `state.life_cycle_state == TERMINATED`. The 300-second ceiling is enough for the cold-start plus retry plus the four task runs.

In [ ]:
# NBVAL_SKIP
# Task 3: Trigger the run and poll until it terminates.

# YOUR CODE: Call w.jobs.run_now(job_id=JOB_ID). Store the returned
# RunNowResponse's run_id in RUN_ID.

global RUN_ID
# YOUR CODE: RUN_ID = w.jobs.run_now(job_id=JOB_ID).run_id

print(f"Triggered run {RUN_ID}. Asking ingest to fall over on attempt 0...")

deadline = time.time() + 600  # 10 minutes total for the retry plus four tasks
final_run = None
while time.time() < deadline:
    info = w.jobs.get_run(RUN_ID)
    life_cycle = (info.state.life_cycle_state.value if info.state and info.state.life_cycle_state else "") or ""
    result_state = (info.state.result_state.value if info.state and info.state.result_state else "") or ""
    if life_cycle.upper() in ("TERMINATED", "INTERNAL_ERROR", "SKIPPED"):
        final_run = info
        break
    time.sleep(10)

if final_run is None:
    print("Run did not terminate within the polling window. Re-run this cell.")
else:
    life_cycle = (final_run.state.life_cycle_state.value if final_run.state.life_cycle_state else "")
    result_state = (final_run.state.result_state.value if final_run.state.result_state else "")
    print(f"Run terminal state: life_cycle={life_cycle} result_state={result_state}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Latest run reached TERMINATED with result_state=SUCCESS.
# Ingest task ran twice (attempt_number=2 or two attempt entries). Notify
# task ran successfully. notify_log has at least one row.


def checkpoint_3(session):
    try:
        if JOB_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="JOB_ID is None.",
            )
        runs = w.jobs.list_runs(job_id=JOB_ID, limit=5)
        run_list = list(runs.runs or [])
        if not run_list:
            return CheckpointResult(
                status="fail",
                error_reason="Job has no runs. Trigger run_now and wait.",
            )
        latest_run = run_list[0]
        info = w.jobs.get_run(latest_run.run_id)

        life_cycle = (info.state.life_cycle_state.value if info.state and info.state.life_cycle_state else "")
        result_state = (info.state.result_state.value if info.state and info.state.result_state else "")
        if (life_cycle or "").upper() != "TERMINATED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest run life_cycle_state={life_cycle!r}; expected TERMINATED. "
                    f"Wait for the run to complete and re-check."
                ),
            )
        if (result_state or "").upper() != "SUCCESS":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest run result_state={result_state!r}; expected SUCCESS. "
                    f"The retry should have succeeded on attempt 1."
                ),
            )

        # Validate ingest had two attempts (retry visible). Different SDK
        # shapes surface this differently: prefer `attempt_number` on the run
        # task, fall back to `attempts` count via list_runs with include_history.
        ingest_run = None
        for t in (info.tasks or []):
            if t.task_key == "ingest":
                ingest_run = t
                break
        if ingest_run is None:
            return CheckpointResult(
                status="fail",
                error_reason="Latest run is missing the ingest task.",
            )
        attempt_number = ingest_run.attempt_number if getattr(ingest_run, "attempt_number", None) is not None else 0
        if attempt_number < 1:
            # Some API shapes report attempt_number as 0 for the original
            # attempt and 1 for the first retry; the retry succeeded means
            # >= 1. Accept both definitions of "two attempts" by checking
            # job's run-history breakdown for ingest task.
            try:
                history = w.jobs.list_runs(job_id=JOB_ID, run_type="JOB_RUN", limit=20)
                ingest_attempts = 0
                for r in (history.runs or []):
                    if r.run_id != info.run_id:
                        continue
                    rinfo = w.jobs.get_run(r.run_id)
                    for tt in (rinfo.tasks or []):
                        if tt.task_key == "ingest":
                            ingest_attempts += 1
                if ingest_attempts < 1:
                    raise RuntimeError("history walk inconclusive")
            except Exception:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"ingest task attempt_number={attempt_number}; expected at "
                        f"least 1 (one retry after the deliberate failure)."
                    ),
                )

        # Notify_log has a new row.
        notify_count_rows = run_sql(f"SELECT count(*) FROM {NOTIFY_FQN}")["rows"]
        notify_count = int(notify_count_rows[0][0]) if notify_count_rows else 0
        if notify_count < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"notify_log has {notify_count} rows; expected at least 1. "
                    f"The condition_task may not have evaluated TRUE, or the "
                    f"notify task may not depend on the TRUE branch."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names which thing failed: run not yet TERMINATED, result_state wrong, ingest did not retry, or notify did not fire. If the run is still running, wait and re-run the checkpoint.

</details>

<details><summary>Hint 2 (direction)</summary>

One SDK call to trigger: `w.jobs.run_now(job_id=JOB_ID).run_id`. Then poll `w.jobs.get_run(run_id)` and check `info.state.life_cycle_state` until it is TERMINATED. The result_state should be SUCCESS because the retry succeeds.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
global RUN_ID
RUN_ID = w.jobs.run_now(job_id=JOB_ID).run_id

deadline = time.time() + 600
while time.time() < deadline:
    info = w.jobs.get_run(RUN_ID)
    if info.state and info.state.life_cycle_state:
        if info.state.life_cycle_state.value == "TERMINATED":
            break
    time.sleep(10)
```

If notify_log is empty, your notify task does not have `outcome="true"` on the `TaskDependency` for `notify_on_low_count`. Re-create the job (or update it) with the correct outcome.

</details>

**Wallet check.** Still at $0.00. One full job run including the deliberate retry takes 3 to 6 minutes of serverless time. If you re-run debugging three times you have used a meaningful slice of the daily quota; cleanup at the end releases everything.

## Task 4: Confirm run history surfaces the retry and the branch decision

The validator inspects the run-history view via `list_runs` and asserts:

- The latest run has a non-null `start_time` and `end_time` (the basics).
- The latest run's `tasks` array includes all four task names (ingest, transform, validate, notify_on_low_count, notify).
- `notify_log` has exactly one row dated within the last hour.
- A second `list_runs` call with `include_history=True` returns at least one extra entry for ingest's first failed attempt OR ingest's `attempt_number` on the terminal run is >= 1.

Nothing to write here; the checkpoint passes if Task 3's run completed cleanly.

In [ ]:
# NBVAL_SKIP
# Task 4: No new code. Print the run-history details for context.

if JOB_ID:
    runs = list(w.jobs.list_runs(job_id=JOB_ID, limit=3).runs or [])
    print(f"Recent runs: {len(runs)}")
    for r in runs:
        ls = (r.state.life_cycle_state.value if r.state and r.state.life_cycle_state else "")
        rs = (r.state.result_state.value if r.state and r.state.result_state else "")
        print(f"  run_id={r.run_id} life_cycle={ls} result={rs}")
else:
    print("JOB_ID is None; re-run Task 2.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Latest run exposes start_time, end_time, four task entries,
# and the notify_log row count is exactly 1.


def checkpoint_4(session):
    try:
        if JOB_ID is None:
            return CheckpointResult(
                status="fail",
                error_reason="JOB_ID is None.",
            )
        runs = list(w.jobs.list_runs(job_id=JOB_ID, limit=3).runs or [])
        if not runs:
            return CheckpointResult(
                status="fail",
                error_reason="Job has no runs.",
            )
        latest = runs[0]
        info = w.jobs.get_run(latest.run_id)

        if not info.start_time:
            return CheckpointResult(
                status="fail",
                error_reason="Latest run is missing start_time.",
            )
        if not info.end_time:
            return CheckpointResult(
                status="fail",
                error_reason="Latest run is missing end_time. Wait for the run to terminate.",
            )

        task_keys = {t.task_key for t in (info.tasks or [])}
        expected_keys = {
            "ingest", "transform", "validate", "notify_on_low_count", "notify",
        }
        missing = expected_keys - task_keys
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run history is missing task entries: {sorted(missing)}. "
                    f"The DAG did not execute all tasks."
                ),
            )

        notify_count_rows = run_sql(f"SELECT count(*) FROM {NOTIFY_FQN}")["rows"]
        notify_count = int(notify_count_rows[0][0]) if notify_count_rows else 0
        if notify_count < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"notify_log has {notify_count} rows; expected at least 1."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint inspects what the run history view already shows. If it complains about missing task entries, your DAG never reached those tasks. If it complains about notify_log, the conditional branch did not evaluate TRUE.

</details>

<details><summary>Hint 2 (direction)</summary>

No code to write. If the validator fails, the fix is back in Task 2 (DAG shape) or Task 3 (re-trigger after fixing). Use `w.jobs.list_runs(job_id=JOB_ID, limit=3)` to inspect the history yourself.

</details>

<details><summary>Hint 3 (spoiler)</summary>

If notify_log has zero rows, the most common cause is that the `notify` task depends on `notify_on_low_count` without `outcome="true"`. Fix:

```python
w.jobs.update(
    job_id=JOB_ID,
    new_settings=j_svc.JobSettings(
        name=JOB_NAME,
        tasks=[ingest_task, transform_task, validate_task, notify_branch_task,
               j_svc.Task(
                   task_key="notify",
                   notebook_task=j_svc.NotebookTask(notebook_path=NOTIFY_NB),
                   depends_on=[j_svc.TaskDependency(
                       task_key="notify_on_low_count", outcome="true"
                   )],
               )],
        tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    ),
)
```

Then re-trigger.

</details>

**Wallet check.** Still at $0.00. The validator runs metadata queries and one SELECT count(*); nothing new on serverless.

## Cleanup

Time to tear it all down. The cell below cancels any in-flight job runs, deletes the Lakeflow Job, drops the three tables, clears the volume, deletes the workspace folder with the four notebooks, then drops the schema with `CASCADE`. The job is critical because in-flight runs consume daily quota.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST. Job is critical.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and getattr(r, "critical", False)]
standard_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and not getattr(r, "critical", False)]
critical_destroyed = len(critical_resources)
standard_destroyed = len(standard_resources)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** One Lakeflow Job, four task notebooks, three managed tables, one UC Volume, one workspace folder. Free Edition swallowed a meaningful chunk of today's quota across the deliberate-fail run plus the retry. The expensive thing this session would have prevented is the team's Tuesday-morning fire drill becoming a quarterly habit.

## Reflection

These are not graded. They are for you.

1. Walk through what Lakeflow Jobs does when a task fails and `max_retries=2`. Name each step: the failure event, the backoff wait, the retry attempt, the run-level state changes during the wait. Why is exponential backoff the default vs a fixed interval?

2. Your team has a downstream Lakeflow Job that depends on this job's success and currently runs on a separate cron. Sketch the trade-offs of (a) keeping the cron and adding a `table_update` trigger, (b) refactoring into a single multi-task Lakeflow Job, (c) using Lakeflow Job runs as the cross-job signaling mechanism. When is each pattern right?

## Exam-Style Practice

**Question 1 (Domain 4, trigger types):**

A Lakeflow Job processes daily batch files that arrive in a UC Volume. The team wants the job to start as soon as the daily file arrives, with no fixed schedule. Which trigger type fits?

A. A `scheduled` trigger with a daily cron at 3am UTC.
B. A `file_arrival` trigger pointed at the UC Volume path the file lands in.
C. A `table_update` trigger pointed at the bronze table the job writes to.
D. A `continuous` trigger so the job is always running.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: a cron schedule assumes the file arrives reliably at a known time; the scenario says "as soon as the daily file arrives," which is event-driven, not time-driven.
- B is correct: file_arrival triggers fire the job when new files appear in the target path. This is the documented Lakeflow Jobs trigger for cloud-storage event-driven workloads, and it fits "as soon as the file arrives" with low latency.
- C is wrong: `table_update` triggers fire on writes to the table; the trigger should fire on the source file arrival, not on a downstream write.
- D is wrong: continuous mode is for long-running streaming workloads, not for daily batch.

</details>

**Question 2 (Domain 4 / 6, retry policy semantics):**

A Lakeflow Job task is configured with `max_retries=3` and `min_retry_interval_millis=10000`. The task fails on its first attempt. After the first failure, when does Lakeflow attempt the next run, and how many total attempts will it make if every attempt fails?

A. Immediately retry; total 1 attempt because `max_retries=3` counts the original attempt.
B. Wait 10 seconds, then attempt 2; if fails wait ~20 seconds for attempt 3; if fails wait ~40 seconds for attempt 4. Total 4 attempts.
C. Wait 10 seconds, then retry up to 3 times with the same 10-second backoff. Total 4 attempts.
D. Wait 30 seconds, then retry up to 3 times with the same 30-second backoff. Total 4 attempts.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: `max_retries=3` means three retries in addition to the original attempt, totaling 4 attempts max if every retry fails.
- B is correct: Lakeflow Jobs use exponential backoff with `min_retry_interval_millis` as the starting interval. The first retry waits ~10 seconds, the second ~20, the third ~40. The total attempt count is 1 (original) + 3 (retries) = 4.
- C is wrong: Lakeflow Jobs retries are exponential by default, not fixed interval.
- D is wrong: the starting interval is what the engineer set (10 seconds), not 30. The 30-second value would come from `min_retry_interval_millis=30000`.

</details>